In [0]:
# The purpose of this notebook is to build our training dataset for the model. This dataset will
# be the basis for our EDA, feature engineering, development of prototype models, etc.
# It will be constructed using a fixed initial window of the data with defined dates.
# We will manually adjust the dates when we want to increase the size of the development period, but
# we will always leave a gap between the end of development and the end of the dataset to ensure
# that we can evaluate generalization. We will fit models using time series cv


from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

# New York midnight boundaries, expressed in UTC.
DEV_START = "2026-08-21T04:00:00Z"
DEV_END = "2026-09-27T04:00:00Z"  # Update to change dates

status = (
    spark.table("citibike_project.citibike.silver_station_status")
    .filter(
        (F.col("fetched_at") >= F.lit(DEV_START).cast("timestamp"))
        & (F.col("fetched_at") < F.lit(DEV_END).cast("timestamp"))
    )
)

# Retain earlier metadata for historical as-of joins.
station_info_raw = (
    spark.table("citibike_project.citibike.silver_station_info")
    .filter(F.col("fetched_at") < F.lit(DEV_END).cast("timestamp"))
)

status.write.mode("overwrite").saveAsTable(
    "citibike_project.citibike.dev_station_status"
)

station_info_raw.write.mode("overwrite").saveAsTable(
    "citibike_project.citibike.dev_station_info"
)